# PERF-INPUT. DataLoader, Packing and Input Movement | 数据加载、装箱与输入搬运性能

**难度：** Hard | **环境：** CPU 机制练习 + 单卡 GPU 机制实验
**标签：** `数据加载器` `装箱` `主机到设备搬运` `输入流水线` | **目标人群：** 训练性能与性能分析学习者

---

## 本节导读

GPU 训练步骤不只包含模型计算。样本读取、批次整理、填充或装箱、主机缓冲和主机到设备搬运共同组成输入流水线；当供给速度低于 GPU 消费速度时，计算设备会等待数据。

本节先建立输入对象的生命周期，再区分有效 Token 与无效填充的成本，最后判断预取、锁页内存和异步搬运在什么条件下能减少暴露等待。


## 前置阅读

- [Part 00 · 12 数据接口与训练生命周期](../00_Prerequisites/12_PyTorch_Minimal_Training_Interface.ipynb)：理解 Dataset、DataLoader 与训练步骤的连接。
- [Part 00 · 17 PyTorch Profiling 基础](../00_Prerequisites/17_PyTorch_Profiling_Basics.ipynb)：区分 CPU 活动、CUDA 活动与等待。
- [Part 01 · 07 CPU/GPU 异构调度](../01_Hardware_Math_and_Systems/07_CPU_GPU_Heterogeneous_Scheduling.ipynb)：理解主机准备、设备执行与数据搬运。
---


### Step 1：输入对象如何到达 GPU

一个训练批次通常经历样本读取、解码或分词、批次整理、主机缓冲、主机到设备搬运，最后才被模型消费。流水线吞吐由最慢的持续阶段限制；队列只能吸收短时抖动，不能消除长期供给不足。

| 阶段 | 主要对象 | 典型资源 | 可观察证据 |
| --- | --- | --- | --- |
| 读取与预处理 | 原始样本、Token 序列 | 存储、CPU、内存 | 取批等待、工作进程忙闲与队列深度 |
| 批次整理 | 填充或装箱后的张量 | CPU、主机内存 | 有效 Token 比例与形状分布 |
| 主机缓冲与搬运 | 锁页或普通内存张量 | 内存带宽、PCIe | 搬运时间、并行区间与同步点 |
| GPU 消费 | 设备张量 | GPU 计算与显存 | 计算内核区间、设备空闲与吞吐 |

![训练输入流水线与背压传播](../docs/public/02_PyTorch_Algorithms/perf_input_pipeline_map.svg)


### Step 2：Packing 改变的是有效工作比例

Padding 将不同长度样本补到统一长度，形状规则但会搬运和计算无效 Token。Packing 把多个短样本放入固定容量容器，以边界信息换取更高的有效 Token 比例。它不会减少真实 Token 数，也不能突破单条样本超过容器容量的约束。

| 量 | 定义 | 变化意味着什么 |
| --- | --- | --- |
| 有效 Token | 原始样本长度之和 | 真正参与目标计算的数据量 |
| 容器 Token | 所有 pack 容量之和 | 实际分配、搬运和计算的上界 |
| Packing 效率 | 有效 Token / 容器 Token | 越接近 1，无效 padding 越少 |
| 样本边界 | pack 内各样本的位置范围 | 必须防止不同样本的目标或注意力串扰 |


### Step 3：预取与异步搬运何时减少等待

预取把下一批的 CPU 准备移到当前批计算期间；锁页内存使主机缓冲可参与异步搬运；独立的搬运资源让复制有机会与计算并发。重叠仍必须保留消费依赖：本批数据完成搬运后，设备计算才能读取它。

连续批次的时间由三条递推关系决定：准备开始时间受上一批准备结束时间约束；复制开始时间受本批准备完成和上一批复制完成约束；计算开始时间同时受本批复制完成和上一批计算完成约束。相邻计算阶段之间的空隙，才是暴露给训练步骤的输入等待。

| 条件 | 未满足时 | 满足后的作用 |
| --- | --- | --- |
| 后续批次提前准备 | GPU 计算结束后才开始取批 | CPU 准备可与当前计算并行 |
| 主机缓冲支持异步搬运 | 搬运无法脱离主机等待 | 主机可继续准备后续批次 |
| 搬运与计算使用独立资源 | 两个阶段仍按顺序串行 | 两个阶段可以并发执行 |
| 消费前保留数据依赖 | 设备可能读取未完成的数据 | 在不使用全局同步的情况下保证正确性 |

![串行输入路径与流水重叠路径](../docs/public/02_PyTorch_Algorithms/perf_input_overlap_timeline.svg)


### Step 4：实现输入容量账本与流水线时间模型

题目区使用样本长度和阶段耗时作为确定性输入：先生成不拆分样本的装箱计划，再建立有效 Token 账本，最后递推多个批次的准备、搬运和计算时间。

| TODO / 函数 | 学习者完成的机制 | 必须满足的约束 | 测试证据 |
| --- | --- | --- | --- |
| TODO 1：更新 pack 状态<br>`build_greedy_packs` | 将当前样本加入 pack 并同步累计长度 | 不拆分样本；顺序不变；超长样本报错 | 容量、覆盖、尾 pack 与失败输入 |
| TODO 2：建立 Token 守恒账本<br>`packing_accounting` | 计算有效、容器、填充 Token 与效率 | 有效量加填充量等于容器量；空输入效率为 1 | 非零填充、空输入与守恒断言 |
| TODO 3：推进输入流水线<br>`schedule_input_pipeline` | 递推准备、H2D 与计算的开始时间 | 四项异步条件缺一时不得声明复制与计算重叠 | 串行、完整重叠、输入饥饿与非法时间 |


In [ ]:
import math


In [ ]:
def build_greedy_packs(lengths: list[int], capacity: int) -> list[list[int]]:
    """按输入顺序生成 pack 长度计划，不拆分样本。"""
    if capacity <= 0:
        raise ValueError("capacity 必须为正数")
    if any((not isinstance(length, int)) or length <= 0 for length in lengths):
        raise ValueError("样本长度必须为正整数")
    if any(length > capacity for length in lengths):
        raise ValueError("存在超过 pack 容量的样本")
    packs, current, current_tokens = [], [], 0
    for length in lengths:
        if current and current_tokens + length > capacity:
            packs.append(current)
            current, current_tokens = [], 0
        # ==========================================
        # TODO 1：更新 pack 状态
        # ==========================================
        # current = ???  # 加入当前样本后的长度列表
        # current_tokens = ???  # 当前 pack 已使用的 Token 数
        raise NotImplementedError("TODO 1：请更新 pack 状态")
    if current:
        packs.append(current)
    return packs


def packing_accounting(packs: list[list[int]], capacity: int) -> dict[str, float]:
    """计算 packing 的 Token 守恒账本。"""
    if capacity <= 0 or any(sum(pack) > capacity for pack in packs):
        raise ValueError("容量非法或 pack 超过容量")
    # ==========================================
    # TODO 2：建立 Token 守恒账本
    # ==========================================
    # useful_tokens = ???  # 样本的真实 Token 总数
    # container_tokens = ???  # pack 数乘以单 pack 容量
    # padding_tokens = ???  # 未被真实 Token 使用的数量
    # efficiency = ???  # 有效 Token 比例；空输入为 1.0
    raise NotImplementedError("TODO 2：请建立 Token 守恒账本")
    return dict(useful_tokens=useful_tokens, container_tokens=container_tokens,
                padding_tokens=padding_tokens, efficiency=efficiency)


def schedule_input_pipeline(
    prepare_times_ms: list[float],
    h2d_times_ms: list[float],
    compute_times_ms: list[float],
    *,
    prefetch: bool,
    pinned: bool,
    non_blocking: bool,
    separate_copy_stream: bool,
) -> list[dict[str, float]]:
    """递推连续批次的准备、H2D 与计算时间线。"""
    if not prepare_times_ms or not (len(prepare_times_ms) == len(h2d_times_ms) == len(compute_times_ms)):
        raise ValueError("三个阶段必须提供等长的非空时间序列")
    if any(value < 0 for values in (prepare_times_ms, h2d_times_ms, compute_times_ms) for value in values):
        raise ValueError("阶段时间不能为负数")
    overlap_enabled = prefetch and pinned and non_blocking and separate_copy_stream
    prepare_end = copy_end = compute_end = 0.0
    timeline = []
    for prepare_ms, h2d_ms, compute_ms in zip(prepare_times_ms, h2d_times_ms, compute_times_ms):
        previous_compute_end = compute_end
        # ==========================================
        # TODO 3：推进输入流水线
        # ==========================================
        # prepare_start = ???  # 是否允许下一批准备与当前计算重叠
        # copy_start = ???  # 等待本批准备完成，并服从复制流顺序
        # compute_start = ???  # 等待本批复制和上一批计算完成
        raise NotImplementedError("TODO 3：请推进输入流水线")
        prepare_end = prepare_start + prepare_ms
        copy_end = copy_start + h2d_ms
        compute_end = compute_start + compute_ms
        timeline.append({
            "prepare_start_ms": prepare_start, "prepare_end_ms": prepare_end,
            "h2d_start_ms": copy_start, "h2d_end_ms": copy_end,
            "compute_start_ms": compute_start, "compute_end_ms": compute_end,
            "exposed_wait_ms": max(0.0, compute_start - previous_compute_end),
        })
    return timeline


### 测试

运行下方测试，检查 packing 的顺序与容量约束、Token 账本守恒，以及输入重叠的必要条件。


In [ ]:
def test_greedy_packs_and_invalid_lengths():
    """装箱结果必须保持顺序、保留尾包并拒绝非法样本长度。"""
    lengths = [6, 3, 7, 2, 1]
    packs = build_greedy_packs(lengths, 10)
    assert packs == [[6, 3], [7, 2, 1]]
    assert [item for pack in packs for item in pack] == lengths
    for invalid in ([0, 3], [11]):
        try:
            build_greedy_packs(invalid, 10)
        except ValueError:
            pass
        else:
            raise AssertionError("非法样本长度必须报错")


def test_accounting_and_padding():
    """有效、填充与容器 Token 必须守恒，并覆盖非零填充。"""
    record = packing_accounting([[6, 3], [7]], 10)
    assert record == {"useful_tokens": 16, "container_tokens": 20,
                      "padding_tokens": 4, "efficiency": 0.8}
    assert record["useful_tokens"] + record["padding_tokens"] == record["container_tokens"]
    assert packing_accounting([], 10)["efficiency"] == 1.0


def test_serial_and_overlapped_pipeline():
    """串行路径保留输入等待，完整异步条件可隐藏第二批输入。"""
    serial = schedule_input_pipeline([4, 4], [2, 2], [10, 10],
        prefetch=False, pinned=True, non_blocking=True, separate_copy_stream=True)
    overlap = schedule_input_pipeline([4, 4], [2, 2], [10, 10],
        prefetch=True, pinned=True, non_blocking=True, separate_copy_stream=True)
    assert [item["compute_start_ms"] for item in serial] == [6.0, 22.0]
    assert [item["exposed_wait_ms"] for item in serial] == [6.0, 6.0]
    assert [item["compute_start_ms"] for item in overlap] == [6.0, 16.0]
    assert overlap[1]["exposed_wait_ms"] == 0.0


def test_input_starvation_and_missing_stream():
    """准备过慢仍会暴露等待，缺少独立复制流时不得声明重叠。"""
    starved = schedule_input_pipeline([12, 12], [5, 5], [10, 10],
        prefetch=True, pinned=True, non_blocking=True, separate_copy_stream=True)
    serialized = schedule_input_pipeline([4, 4], [2, 2], [10, 10],
        prefetch=True, pinned=True, non_blocking=True, separate_copy_stream=False)
    assert starved[1]["exposed_wait_ms"] == 2.0
    assert serialized[1]["exposed_wait_ms"] == 6.0
    try:
        schedule_input_pipeline([1], [1, 2], [1], prefetch=True, pinned=True,
                                non_blocking=True, separate_copy_stream=True)
    except ValueError:
        pass
    else:
        raise AssertionError("阶段时间序列长度不一致时必须报错")


test_greedy_packs_and_invalid_lengths()
test_accounting_and_padding()
test_serial_and_overlapped_pipeline()
test_input_starvation_and_missing_stream()
print("All tests passed!")


---

🛑 **STOP HERE** 🛑
<br><br><br><br><br><br><br><br><br><br>
> 请先尝试自己完成代码并跑通测试。<br>
> 如果遇到困难没有思路，可以向下滚动查看参考答案。
<br><br><br><br><br><br><br><br><br><br>

---


## 参考代码与解析

### 代码


In [ ]:
def build_greedy_packs(lengths: list[int], capacity: int) -> list[list[int]]:
    """按输入顺序生成 pack 长度计划，不拆分样本。"""
    if capacity <= 0:
        raise ValueError("capacity 必须为正数")
    if any((not isinstance(length, int)) or length <= 0 for length in lengths):
        raise ValueError("样本长度必须为正整数")
    if any(length > capacity for length in lengths):
        raise ValueError("存在超过 pack 容量的样本")
    packs, current, current_tokens = [], [], 0
    for length in lengths:
        if current and current_tokens + length > capacity:
            packs.append(current)
            current, current_tokens = [], 0
        # TODO 1：更新 pack 状态。
        current.append(length)
        current_tokens += length
    if current:
        packs.append(current)
    return packs


def packing_accounting(packs: list[list[int]], capacity: int) -> dict[str, float]:
    """计算 packing 的 Token 守恒账本。"""
    if capacity <= 0 or any(sum(pack) > capacity for pack in packs):
        raise ValueError("容量非法或 pack 超过容量")
    # TODO 2：建立 Token 守恒账本。
    useful_tokens = sum(sum(pack) for pack in packs)
    container_tokens = len(packs) * capacity
    padding_tokens = container_tokens - useful_tokens
    efficiency = useful_tokens / container_tokens if container_tokens else 1.0
    return dict(useful_tokens=useful_tokens, container_tokens=container_tokens,
                padding_tokens=padding_tokens, efficiency=efficiency)


def schedule_input_pipeline(
    prepare_times_ms: list[float], h2d_times_ms: list[float], compute_times_ms: list[float], *,
    prefetch: bool, pinned: bool, non_blocking: bool, separate_copy_stream: bool,
) -> list[dict[str, float]]:
    """递推连续批次的准备、H2D 与计算时间线。"""
    if not prepare_times_ms or not (len(prepare_times_ms) == len(h2d_times_ms) == len(compute_times_ms)):
        raise ValueError("三个阶段必须提供等长的非空时间序列")
    if any(value < 0 for values in (prepare_times_ms, h2d_times_ms, compute_times_ms) for value in values):
        raise ValueError("阶段时间不能为负数")
    overlap_enabled = prefetch and pinned and non_blocking and separate_copy_stream
    prepare_end = copy_end = compute_end = 0.0
    timeline = []
    for prepare_ms, h2d_ms, compute_ms in zip(prepare_times_ms, h2d_times_ms, compute_times_ms):
        previous_compute_end = compute_end
        # TODO 3：推进输入流水线。
        prepare_start = prepare_end if overlap_enabled else max(prepare_end, compute_end)
        copy_start = max(prepare_start + prepare_ms, copy_end) if overlap_enabled else max(prepare_start + prepare_ms, copy_end, compute_end)
        compute_start = max(copy_start + h2d_ms, compute_end)
        prepare_end = prepare_start + prepare_ms
        copy_end = copy_start + h2d_ms
        compute_end = compute_start + compute_ms
        timeline.append({
            "prepare_start_ms": prepare_start, "prepare_end_ms": prepare_end,
            "h2d_start_ms": copy_start, "h2d_end_ms": copy_end,
            "compute_start_ms": compute_start, "compute_end_ms": compute_end,
            "exposed_wait_ms": max(0.0, compute_start - previous_compute_end),
        })
    return timeline


### 解析

#### TODO 1：更新 pack 状态

- 当前 pack 放不下新样本时先封口，再把样本放入新 pack。
- 列表和累计 Token 必须同步更新；测试同时检查顺序、容量、尾包和非法长度。

#### TODO 2：建立 Token 守恒账本

- 有效 Token 来自样本长度总和，容器 Token 来自 pack 数与固定容量。
- 填充量等于容器量减有效量；测试使用非零填充，避免只验证恰好装满的特例。

#### TODO 3：推进输入流水线

- 准备、复制和计算各自服从前序依赖；计算只能在本批复制完成且上一批计算结束后开始。
- 只有预取、锁页内存、非阻塞搬运和独立复制流同时成立时，准备与复制才可进入重叠路径。测试分别覆盖串行、完整隐藏和输入饥饿。


### Step 5：GPU 机制实验——输入供给矩阵

CPU 题目确认容量、Token 守恒与重叠条件；GPU 实验固定合成张量和计算负载，对比单进程普通内存、单进程锁页内存和多工作进程预取三条输入路径。

| 学习问题 | CPU 已确认 | GPU 要确认与后续使用 |
| --- | --- | --- |
| 输入配置是否减少真实设备等待 | 装箱效率与暴露等待模型 | 取批等待、主机到设备搬运、步骤中位数与 P95、吞吐和峰值显存；结果可进入训练性能候选复测 |

#### 5.1 环境与固定工作负载

本实验使用 `PERF-CUDA-MECHANISM`。首次运行按[性能分析实验 Quickstart](../topic_discussion/performance_analysis/quickstart.md#perf-cuda-mechanism-quickstart)创建并预检环境，本页的输入、对照和结果路径登记在[性能优化实验资产表](../topic_discussion/performance_optimization/experiment_assets.md)。本页使用固定合成数据，不下载模型或数据集。

| 配置层 | 本节默认口径 | 修改规则 |
| --- | --- | --- |
| 运行环境 | 单张 CUDA GPU；`PERF-CUDA-MECHANISM` | 环境预检未返回 `status=ok` 时停止 |
| 工作负载 | 批量 8；序列 128；特征 512；固定 80 个训练步骤 | 三条路径共享样本、矩阵、数据类型与随机种子 |
| 候选矩阵 | 普通内存 / 锁页内存 / 多工作进程 + 锁页内存 + 预取 | 不把工作进程数与张量形状同时改变 |
| 测量方法 | 取批等待、搬运时间、步骤中位数与 P95、每秒样本数、活动显存峰值 | 先检查校验值，再解释性能 |
| 结果资产 | `benchmarks/results/perf_input_pipeline/<semantic_id>_<run_label>.json` | 正式复测使用唯一标签 |


In [1]:
RUN_MODE = "dry_run"  # 默认值 dry_run；改为 real_gpu 执行真实矩阵。
RUN_LABEL = "default"  # 默认值 default；正式复测改为唯一标签。
PROFILE_ID = "PERF-CUDA-MECHANISM"  # 默认性能机制环境。
SEMANTIC_ID = "PERF-INPUT"  # 稳定机制 ID。
GPU_CONFIG = {
    "batch_size": 8,        # 默认值 8；三条路径保持相同。
    "seq_len": 128,         # 默认值 128；代表 Token 维。
    "feature_dim": 512,     # 默认值 512；控制搬运量与矩阵计算量。
    "warmup_steps": 10,     # 默认值 10；不计入统计。
    "repeat_steps": 80,     # 默认值 80；统计 median 与 P95。
    "candidate_workers": 2, # 默认值 2；多 worker 候选进程数。
    "prefetch_factor": 2,   # 默认值 2；每个 worker 预取批次数。
    "seed": 42,             # 默认值 42；固定样本与权重。
}
RESULTS_RELATIVE_DIR = "benchmarks/results/perf_input_pipeline"


#### 5.2 执行并保存 JSON

下方单元构造固定合成 Dataset 和矩阵计算，对三种 DataLoader 路径分别测量取批、H2D 与完整 step。该实验反映当前主机与 PCIe 条件，不替代真实存储和分词流水线压测。


In [2]:
from pathlib import Path
import json, platform, statistics, time
import torch
from torch.utils.data import DataLoader, Dataset
from tools.project_runtime import resolve_project_root

class FixedTensorDataset(Dataset):
    """使用可复现 CPU 张量模拟已完成解码或分词的样本。"""
    def __init__(self, count, seq_len, feature_dim, seed):
        generator = torch.Generator().manual_seed(seed)
        self.samples = torch.randn(count, seq_len, feature_dim, generator=generator)
    def __len__(self):
        return self.samples.shape[0]
    def __getitem__(self, index):
        return self.samples[index]

def _p95(values):
    ordered = sorted(values)
    return ordered[int(0.95 * (len(ordered) - 1))]

def _run_path(dataset, weight, spec):
    kwargs = dict(batch_size=GPU_CONFIG["batch_size"], shuffle=False,
                  num_workers=spec["workers"], pin_memory=spec["pin_memory"], drop_last=True)
    if spec["workers"] > 0:
        kwargs.update(prefetch_factor=GPU_CONFIG["prefetch_factor"], persistent_workers=True)
    iterator = iter(DataLoader(dataset, **kwargs))
    waits, copies, steps, checksum = [], [], [], 0.0
    total = GPU_CONFIG["warmup_steps"] + GPU_CONFIG["repeat_steps"]
    torch.cuda.reset_peak_memory_stats()
    for step in range(total):
        wall_start = time.perf_counter()
        batch = next(iterator)
        after_next = time.perf_counter()
        begin, end = torch.cuda.Event(True), torch.cuda.Event(True)
        begin.record()
        device_batch = batch.to("cuda", non_blocking=spec["non_blocking"])
        end.record()
        checksum += float(torch.matmul(device_batch, weight).mean().detach().cpu())
        torch.cuda.synchronize()
        if step >= GPU_CONFIG["warmup_steps"]:
            waits.append((after_next - wall_start) * 1000)
            copies.append(begin.elapsed_time(end))
            steps.append((time.perf_counter() - wall_start) * 1000)
    elapsed = sum(steps) / 1000
    return {
        **spec, "batch_wait_median_ms": statistics.median(waits),
        "h2d_median_ms": statistics.median(copies),
        "step_median_ms": statistics.median(steps), "step_p95_ms": _p95(steps),
        "samples_per_second": GPU_CONFIG["repeat_steps"] * GPU_CONFIG["batch_size"] / elapsed,
        "peak_allocated_mb": torch.cuda.max_memory_allocated() / 2**20, "checksum": checksum,
    }

REPO_ROOT = resolve_project_root()
RESULTS_DIR = REPO_ROOT / RESULTS_RELATIVE_DIR
if RUN_MODE == "real_gpu":
    if not torch.cuda.is_available():
        raise RuntimeError('需要 CUDA GPU；无 GPU 时保持 RUN_MODE="dry_run"。')
    torch.manual_seed(GPU_CONFIG["seed"])
    count = (GPU_CONFIG["warmup_steps"] + GPU_CONFIG["repeat_steps"]) * GPU_CONFIG["batch_size"]
    dataset = FixedTensorDataset(count, GPU_CONFIG["seq_len"], GPU_CONFIG["feature_dim"], GPU_CONFIG["seed"])
    weight = torch.randn(GPU_CONFIG["feature_dim"], GPU_CONFIG["feature_dim"], device="cuda")
    matrix = {
        "pageable_single": {"workers": 0, "pin_memory": False, "non_blocking": False},
        "pinned_single": {"workers": 0, "pin_memory": True, "non_blocking": True},
        "prefetched_pinned": {"workers": GPU_CONFIG["candidate_workers"], "pin_memory": True, "non_blocking": True},
    }
    metrics = {name: _run_path(dataset, weight, spec) for name, spec in matrix.items()}
    base, candidate = metrics["pageable_single"], metrics["prefetched_pinned"]
    diff = abs(candidate["checksum"] - base["checksum"]) / max(abs(base["checksum"]), 1e-12)
    passed = diff <= 1e-5
    record = {
        "schema_version": "performance-mechanism-gpu/v1", "profile_id": PROFILE_ID,
        "semantic_id": SEMANTIC_ID, "run_label": RUN_LABEL,
        "framework": {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda},
        "hardware": {"device": torch.cuda.get_device_name(0),
                     "total_memory_bytes": torch.cuda.get_device_properties(0).total_memory},
        "workload": GPU_CONFIG, "metrics": metrics,
        "correctness": {"checksum_relative_diff": diff, "passed": passed},
        "failure": None, "evidence_level": "local_mechanism_benchmark",
        "decision": "accept_prefetch_for_throughput_and_tail_latency" if passed and
                    candidate["samples_per_second"] > base["samples_per_second"] and
                    candidate["step_p95_ms"] < base["step_p95_ms"] else "tune_input_pipeline",
    }
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    result_path = RESULTS_DIR / f"{SEMANTIC_ID.lower()}_{RUN_LABEL}.json"
    result_path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(record, ensure_ascii=False, indent=2))
else:
    print({"run_mode": RUN_MODE, "profile_id": PROFILE_ID, "semantic_id": SEMANTIC_ID, "config": GPU_CONFIG})


{
  "schema_version": "performance-mechanism-gpu/v1",
  "profile_id": "PERF-CUDA-MECHANISM",
  "semantic_id": "PERF-INPUT",
  "run_label": "rtx5070ti_input_v1",
  "framework": {
    "python": "3.12.15",
    "torch": "2.11.0+cu128",
    "cuda": "12.8"
  },
  "hardware": {
    "device": "NVIDIA GeForce RTX 5070 Ti Laptop GPU",
    "total_memory_bytes": 12328960000
  },
  "workload": {
    "batch_size": 8,
    "seq_len": 128,
    "feature_dim": 512,
    "warmup_steps": 10,
    "repeat_steps": 80,
    "candidate_workers": 2,
    "prefetch_factor": 2,
    "seed": 42
  },
  "metrics": {
    "pageable_single": {
      "workers": 0,
      "pin_memory": false,
      "non_blocking": false,
      "batch_wait_median_ms": 0.1414399994246196,
      "h2d_median_ms": 0.22934400290250778,
      "step_median_ms": 0.5671580001944676,
      "step_p95_ms": 0.7046999999147374,
      "samples_per_second": 13612.099493722711,
      "peak_allocated_mb": 13.12646484375,
      "checksum": 0.006802883930504322
  

#### 5.3 读取结果

读取同一 `RUN_LABEL` 的结果文件，先确认 checksum 和失败字段，再比较三条路径的取批等待、H2D 与完整 step。


In [3]:
if RUN_MODE == "real_gpu":
    result_path = RESULTS_DIR / f"{SEMANTIC_ID.lower()}_{RUN_LABEL}.json"
    if not result_path.is_file():
        raise FileNotFoundError(f"未找到结果，请先运行 5.2：{result_path}")
    record = json.loads(result_path.read_text(encoding="utf-8"))
    for name, metrics in record["metrics"].items():
        print(f"{name}: wait={metrics['batch_wait_median_ms']:.3f} ms, "
              f"h2d={metrics['h2d_median_ms']:.3f} ms, step={metrics['step_median_ms']:.3f} ms, "
              f"throughput={metrics['samples_per_second']:.1f} samples/s")
    print(f"correctness={record['correctness']} | decision={record['decision']}")
else:
    print("dry_run：没有读取 GPU 结果。")


pageable_single: wait=0.141 ms, h2d=0.229 ms, step=0.567 ms, throughput=13612.1 samples/s
pinned_single: wait=0.139 ms, h2d=0.152 ms, step=0.379 ms, throughput=18842.3 samples/s
prefetched_pinned: wait=0.038 ms, h2d=0.162 ms, step=0.394 ms, throughput=20302.6 samples/s
correctness={'checksum_relative_diff': 0.0, 'passed': True} | decision=accept_prefetch_candidate


#### 5.4 解释证据与决策

RTX 5070 Ti 实测中，三条路径校验值一致。锁页单进程路径获得最低步骤中位数；多工作进程预取路径获得最低等待、最低 P95 和最高吞吐，因此候选选择必须随目标指标变化。

| 观察项 | 读取字段 | 当前结果 | 解释与动作 |
| --- | --- | --- | --- |
| 运行有效性 | `failure`、工作负载字段 | 无失败；三条路径共享输入、矩阵和步骤数 | 当前结果具备同口径比较条件 |
| 语义一致性 | `checksum_relative_diff` | 相对差异 0；通过 | 性能变化不由计算结果变化解释 |
| 输入供给 | `batch_wait_median_ms` | 普通 0.141、锁页 0.139、预取 0.038 ms | 预取将取批等待降低约 73.1% |
| 搬运与中位延迟 | H2D 与步骤中位数 | 锁页 H2D 0.152 ms；步骤中位数 0.379 ms，三者最低 | 延迟优先时选择锁页单进程路径 |
| 尾延迟与吞吐 | 步骤 P95、samples/s | 预取 P95 0.488 ms；20,303 samples/s，三者最佳 | 吞吐和尾延迟优先时选择预取候选；不把它表述为所有指标最优 |


## 相关阅读

- [PyTorch DataLoader 官方文档](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)
- [PyTorch CUDA 语义：异步执行与 pinned memory](https://pytorch.org/docs/stable/notes/cuda.html)
- [性能优化 · 03 训练状态与资源账本](../topic_discussion/performance_optimization/03_training_memory_ledger.md)
- [PERF-IO：模型产物、状态传输与输出回写的 I/O 关键路径](./PERF-IO_Artifact_State_and_Output_IO_Critical_Path.ipynb)
